In [ ]:
# The previous notebook was Example_Data_Creation where we created .pkl's to save our data

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [3]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
import time
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

In [4]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [79]:
# Set environment variables
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create new session with explicit local binding
spark = SparkSession.builder \
    .appName("EEG_Analysis") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .master("local[*]") \
    .getOrCreate()


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


25/04/07 02:52:57 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/04/07 02:52:57 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try:
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context


In [23]:
# This is how we would load the .pkl's back in (it can take a minute so be patient)
# Step 1: Load back into pandas
group_a_pandas_df_loaded = pd.read_pickle("features_alz_example.pkl")
group_c_pandas_df_loaded = pd.read_pickle("features_cntrl_example.pkl")

# Step 2: Convert to Spark DataFrames
group_a_spark_df_loaded = spark.createDataFrame(group_a_pandas_df_loaded)
group_c_spark_df_loaded = spark.createDataFrame(group_c_pandas_df_loaded)

In [24]:
type(group_a_spark_df_loaded)

pyspark.sql.dataframe.DataFrame

In [25]:
#just renaming things now that we understand the types and where things are coming from
alz_df = group_a_spark_df_loaded
cntrl_df = group_c_spark_df_loaded

In [26]:
# Now lets do dimensionality reducton by first normalizing the power and then doing PCA.
# First step is lets split the data into training/testing

In [36]:
from dimensionality_reduction import normalize_power
# *REFERENCE* df_a_norm, df_c_norm = normalize_power(result_group_a, result_group_c)

In [46]:
cntrl_df.columns

['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']

In [44]:
NUM_TEST_SUBJECTS_PER_GROUP = 1  # i know before we had three , but 2 is better 3 took out too much data.

alz_test_subjects = (
    alz_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)


cntrl_test_subjects = (
    cntrl_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)


25/04/07 02:25:17 WARN TaskSetManager: Stage 55 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 02:25:17 WARN TaskSetManager: Stage 58 contains a task of very large size (6561 KiB). The maximum recommended task size is 1000 KiB.


In [45]:
print(f"azl test subjects {alz_test_subjects}\ncntrl test subjects {cntrl_test_subjects}")

azl test subjects ['sub-001']
cntrl test subjects ['sub-037']


In [48]:
# now we put the labels on both datasets , and split it up into testing/training

In [50]:
from pyspark.sql.functions import lit

In [53]:
# making alz have hte label 1 and cntrl 0 for the ml portion ahea

In [54]:
alz_df = alz_df.withColumn("label", lit(1))
cntrl_df = cntrl_df.withColumn("label", lit(0))

In [55]:
# Filter test rows
alz_test_df = alz_df.filter(alz_df.SubjectID.isin(alz_test_subjects))
cntrl_test_df = cntrl_df.filter(cntrl_df.SubjectID.isin(cntrl_test_subjects))

# Filter training rows (not in test subjects)
alz_train_df = alz_df.filter(~alz_df.SubjectID.isin(alz_test_subjects))
cntrl_train_df = cntrl_df.filter(~cntrl_df.SubjectID.isin(cntrl_test_subjects))


In [56]:
train_df = alz_train_df.unionByName(cntrl_train_df)
test_df = alz_test_df.unionByName(cntrl_test_df)


In [58]:
from dimensionality_reduction import normalize_power # it z-scores the data
# NOTE, this uses the first parameters for mean and std for the z-score, so none of test_df's data is used to z-score
train_df, test_df = normalize_power(train_df, test_df) 

In [60]:
from dimensionality_reduction import prepare_features_for_pca
# this pivots the tables so that its better suited for PCA and ML with pyspark's libraries
print(train_df.columns)
train_df, train_features_column = prepare_features_for_pca(train_df)
test_df, test_features_column = prepare_features_for_pca(test_df)
print(train_df.columns) # as we can see after they get flatened 

['Electrode', 'WaveBand', 'SubjectID', 'EpochID', 'Power', 'label']


25/04/07 02:38:02 WARN TaskSetManager: Stage 67 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 02:38:03 WARN TaskSetManager: Stage 70 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 02:38:05 WARN TaskSetManager: Stage 73 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 02:38:06 WARN TaskSetManager: Stage 76 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
[Stage 76:===========================>                           (10 + 10) / 20]

['SubjectID', 'EpochID', 'label', 'T4_Beta', 'Fp2_Beta', 'P3_Alpha', 'O2_Alpha', 'T3_Delta', 'F7_Delta', 'F8_Beta', 'T4_Alpha', 'F3_Delta', 'P4_Theta', 'T3_Beta', 'F7_Theta', 'T5_Theta', 'P4_Alpha', 'T6_Alpha', 'Cz_Alpha', 'T5_Alpha', 'Cz_Delta', 'F4_Alpha', 'Pz_Beta', 'Fp1_Total', 'F4_Theta', 'O2_Beta', 'O1_Beta', 'Pz_Total', 'Pz_Alpha', 'Fp2_Alpha', 'O1_Delta', 'C3_Theta', 'Cz_Theta', 'T5_Total', 'C4_Alpha', 'Cz_Total', 'T4_Total', 'P3_Total', 'Fp1_Delta', 'Fz_Delta', 'Cz_Beta', 'Fz_Total', 'T6_Total', 'F8_Alpha', 'C4_Delta', 'F4_Total', 'O1_Alpha', 'F3_Beta', 'F4_Delta', 'Fp2_Total', 'P3_Beta', 'C4_Theta', 'Pz_Delta', 'P3_Delta', 'Fp1_Beta', 'Fp1_Alpha', 'P4_Delta', 'F7_Total', 'T5_Beta', 'O2_Total', 'F7_Alpha', 'T6_Delta', 'F4_Beta', 'F8_Delta', 'F8_Total', 'O2_Theta', 'P4_Total', 'Fz_Beta', 'C4_Beta', 'T3_Total', 'F8_Theta', 'C3_Total', 'F3_Theta', 'O2_Delta', 'T4_Delta', 'Fp1_Theta', 'Fp2_Theta', 'C3_Delta', 'F3_Alpha', 'P3_Theta', 'F7_Beta', 'P4_Beta', 'O1_Total', 'O1_Theta', 'F

In [77]:
if train_features_column != test_features_column:
    print("!! VERY UNUSUAL, NEED TO DEBUG, it means that the trainig and testing have different columns :( !!")

In [80]:
from dimensionality_reduction import fit_pca_model
#Note , this finds the features to explain the model's PCA
K_VAR_TARGET=0.95
pca_model_func, k_val = fit_pca_model(train_df, train_features_column, variance_target=K_VAR_TARGET)

25/04/07 02:53:06 WARN TaskSetManager: Stage 252 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 02:53:07 WARN TaskSetManager: Stage 255 contains a task of very large size (7920 KiB). The maximum recommended task size is 1000 KiB.
25/04/07 02:53:09 ERROR Executor: Exception in task 3.0 in stage 257.0 (TID 1081)
java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method)
	at java.base/java.io.FileOutputStream.write(FileOutputStream.java:354)
	at org.apache.spark.storage.TimeTrackingOutputStream.write(TimeTrackingOutputStream.java:59)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:81)
	at java.base/java.io.BufferedOutputStream.write(BufferedOutputStream.java:127)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net.jpountz.lz4.LZ4BlockOutputStream.write(LZ4BlockOutputStream.java:178)
	at org.apache.spark.st

Py4JJavaError: An error occurred while calling o988.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 257.0 failed 1 times, most recent failure: Lost task 3.0 in stage 257.0 (TID 1081) (localhost executor driver): java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method)
	at java.base/java.io.FileOutputStream.write(FileOutputStream.java:354)
	at org.apache.spark.storage.TimeTrackingOutputStream.write(TimeTrackingOutputStream.java:59)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:81)
	at java.base/java.io.BufferedOutputStream.write(BufferedOutputStream.java:127)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net.jpountz.lz4.LZ4BlockOutputStream.write(LZ4BlockOutputStream.java:178)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:345)
	at org.apache.spark.util.collection.unsafe.sor

[Stage 243:====================================================>  (19 + 0) / 20]

In [ ]:
print(f"We can explian {target} with {k_val} features. That is a lot less then {len(train_df.columns)-3} (we hope).") #-3 for subjectID , epochID and lebel

[Stage 243:====================================================>  (19 + 0) / 20]

In [ ]:
# know that we know we can explain 95% of the variance (or what we set target to) ,
# lets make our dataframces only have those important columns
from dimensionality_reduction import apply_pca_model
train_df = apply_pca_model(train_df, train_features_column, train_features_column, pca_model_func, k_val)
test_df  = apply_pca_model(test_df, train_features_column, train_features_column, pca_model_func, k_val)